In [1]:
# auto-sklearn Benchmark - Gutenberg Gait Database

# Imports
import os
import time
import json
import warnings
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics
import autosklearn.classification
import autosklearn.regression
import autosklearn.metrics

warnings.filterwarnings('ignore')

In [ ]:
# Konfiguration
PREPARED_DATA_PATH   = 'saved_models/prepared/prepared_data.csv'
SPLIT_META_PATH      = 'saved_models/prepared/split_meta.json'

CLASSIFICATION_TIME  = 300
REGRESSION_TIME      = 300
PER_RUN_TIME_LIMIT   = 120
CV_FOLDS             = 5
MEMORY_LIMIT_MB      = 9000 #Out of all frameworks, auto-sklearn is the most memory-intensive. It might crash, if system memory is low.
RANDOM_STATE         = 42
EXCLUDE_WINDOW       = 10

# Kraftrichtungen, an denen jeweils Regressionen durchgefuehrt werden
# (vertikal, anterior-posterior, medio-lateral).
REGRESSION_DIRECTIONS = ['F_V_PRO_', 'F_AP_PRO_', 'F_ML_PRO_']

# Punkte im Gangzyklus (in % der Kurve), an denen je Richtung eine eigene
# Regression durchgefuehrt wird -> 4 Punkte x 3 Richtungen = 12 Aufgaben.
TARGET_PERCENTAGES    = [20, 40, 60, 80]

# Verzeichnis fuer trainierte Modelle + Train/Test-Daten (spaetere SHAP-Analyse).
SAVED_MODELS_DIR      = 'saved_models/auto-sklearn'

In [3]:
# Classification-Benchmark: Training + Auswertung
def benchmark_autosklearn_classification(X, y, groups, split_col, task_name,
                                          time_left_for_this_task=CLASSIFICATION_TIME,
                                          per_run_time_limit=PER_RUN_TIME_LIMIT,
                                          feature_names=None):
    print(f"\n{'='*60}")
    print(f"[auto-sklearn] Classification Task: {task_name}")
    print(f"{'='*60}")

    # Split anhand der von 00_preprocessing vorgegebenen SPLIT-Spalte
    train_idx = np.where(split_col == 'train')[0]
    test_idx = np.where(split_col == 'test')[0]
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx].astype(int), y[test_idx].astype(int)

    print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
    print(f"Class distribution - Train: {np.bincount(y_train)}, Test: {np.bincount(y_test)}")

    automl = autosklearn.classification.AutoSklearnClassifier(
        time_left_for_this_task=time_left_for_this_task,
        per_run_time_limit=per_run_time_limit,
        n_jobs=-1,
        seed=42,
        memory_limit=MEMORY_LIMIT_MB,
        resampling_strategy='cv',
        resampling_strategy_arguments={'folds': CV_FOLDS},
        metric=autosklearn.metrics.accuracy,
    )

    print("Training auto-sklearn classifier...")
    start_time = time.time()
    automl.fit(X_train, y_train)
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")

    y_pred = automl.predict(X_test)

    accuracy = sklearn.metrics.accuracy_score(y_test, y_pred)
    balanced_accuracy = sklearn.metrics.balanced_accuracy_score(y_test, y_pred)
    f1_macro = sklearn.metrics.f1_score(y_test, y_pred, average='macro')
    f1_weighted = sklearn.metrics.f1_score(y_test, y_pred, average='weighted')

    results = {
        'framework': 'auto-sklearn',
        'task': task_name,
        'task_type': 'classification',
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'training_time_seconds': training_time,
        'n_train': len(X_train),
        'n_test': len(X_test),
        'classes': len(np.unique(y)),
        'split_method': 'precomputed subject-grouped split (00_preprocessing.py)',
        'sprint_statistics': str(automl.sprint_statistics()),
        'show_models': str(automl.show_models()),
    }

    print(f"\nResults for {task_name} (auto-sklearn):")
    print(f"  Accuracy:          {accuracy:.4f}")
    print(f"  Balanced accuracy: {balanced_accuracy:.4f}")
    print(f"  F1 (macro):        {f1_macro:.4f}")
    print(f"  F1 (weighted):     {f1_weighted:.4f}")
    print(f"  Training time:     {training_time:.2f} seconds")
    print("\nEnsemble information:")
    print(automl.sprint_statistics())

    # Modell + Daten fuer spaetere, separate SHAP-Analyse speichern
    if feature_names is not None:
        task_dir = os.path.join(SAVED_MODELS_DIR, task_name)
        os.makedirs(task_dir, exist_ok=True)
        try:
            joblib.dump(automl, os.path.join(task_dir, 'model.joblib'))
        except Exception as e:
            print(f"  WARNUNG: Modell konnte nicht gespeichert werden ({e}). "
                  f"Train/Test-Daten werden trotzdem gesichert.")
        np.save(os.path.join(task_dir, 'X_train.npy'), X_train)
        np.save(os.path.join(task_dir, 'X_test.npy'), X_test)
        np.save(os.path.join(task_dir, 'y_train.npy'), y_train)
        np.save(os.path.join(task_dir, 'y_test.npy'), y_test)
        with open(os.path.join(task_dir, 'feature_names.json'), 'w') as f:
            json.dump(list(feature_names), f)
        with open(os.path.join(task_dir, 'meta.json'), 'w') as f:
            json.dump({'framework': 'auto-sklearn', 'task_name': task_name,
                        'split_source': SPLIT_META_PATH}, f)
        print(f"  Modell + Daten fuer SHAP gespeichert unter: {task_dir}")

    return results

In [ ]:
# Regression-Benchmark: Training + Auswertung
def benchmark_autosklearn_regression(X, y, groups, split_col, task_name,
                                      time_left_for_this_task=REGRESSION_TIME,
                                      per_run_time_limit=PER_RUN_TIME_LIMIT,
                                      feature_names=None):
    print(f"\n{'='*60}")
    print(f"[auto-sklearn] Regression Task: {task_name}")
    print(f"{'='*60}")

    # Split anhand der von 00_preprocessing vorgegebenen SPLIT-Spalte
    train_idx = np.where(split_col == 'train')[0]
    test_idx = np.where(split_col == 'test')[0]
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
    print(f"Target range - Train: [{y_train.min():.3f}, {y_train.max():.3f}], "
          f"Test: [{y_test.min():.3f}, {y_test.max():.3f}]")

    automl = autosklearn.regression.AutoSklearnRegressor(
        time_left_for_this_task=time_left_for_this_task,
        per_run_time_limit=per_run_time_limit,
        n_jobs=-1,
        seed=42,
        memory_limit=MEMORY_LIMIT_MB,
        resampling_strategy='cv',
        resampling_strategy_arguments={'folds': CV_FOLDS},
        metric=autosklearn.metrics.r2,
    )

    print("Training auto-sklearn regressor...")
    start_time = time.time()
    automl.fit(X_train, y_train)
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")

    y_pred = automl.predict(X_test)

    # Performance metriken berechnen
    r2 = sklearn.metrics.r2_score(y_test, y_pred)
    mae = sklearn.metrics.mean_absolute_error(y_test, y_pred)
    mse = sklearn.metrics.mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_test - y_pred) / np.maximum(np.abs(y_test), 1e-10))) * 100
    y_range = np.ptp(y_test)
    nrmse = rmse / y_range if y_range > 0 else np.nan

    results = {
        'framework': 'auto-sklearn',
        'task': task_name,
        'task_type': 'regression',
        'r2': r2,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mape': mape,
        'nrmse': nrmse,
        'training_time_seconds': training_time,
        'n_train': len(X_train),
        'n_test': len(X_test),
        'target_mean': float(np.mean(y)),
        'target_std': float(np.std(y)),
        'split_method': 'precomputed subject-grouped split (00_preprocessing.py)',
        'sprint_statistics': str(automl.sprint_statistics()),
        'show_models': str(automl.show_models()),
    }

    print(f"\nResults for {task_name} (auto-sklearn):")
    print(f"  R2:    {r2:.4f}")
    print(f"  MAE:   {mae:.4f}")
    print(f"  RMSE:  {rmse:.4f}")
    print(f"  MAPE:  {mape:.2f}% ")
    print(f"  NRMSE: {nrmse:.4f}")
    print(f"  Training time: {training_time:.2f} seconds")
    print("\nEnsemble information:")
    print(automl.sprint_statistics())

    # Modell + Daten fuer spaetere, separate SHAP-Analyse speichern
    if feature_names is not None:
        task_dir = os.path.join(SAVED_MODELS_DIR, task_name)
        os.makedirs(task_dir, exist_ok=True)
        try:
            joblib.dump(automl, os.path.join(task_dir, 'model.joblib'))
        except Exception as e:
            print(f"  WARNUNG: Modell konnte nicht gespeichert werden ({e}). "
                  f"Train/Test-Daten werden trotzdem gesichert.")
        np.save(os.path.join(task_dir, 'X_train.npy'), X_train)
        np.save(os.path.join(task_dir, 'X_test.npy'), X_test)
        np.save(os.path.join(task_dir, 'y_train.npy'), y_train)
        np.save(os.path.join(task_dir, 'y_test.npy'), y_test)
        with open(os.path.join(task_dir, 'feature_names.json'), 'w') as f:
            json.dump(list(feature_names), f)
        with open(os.path.join(task_dir, 'meta.json'), 'w') as f:
            json.dump({'framework': 'auto-sklearn', 'task_name': task_name,
                        'split_source': SPLIT_META_PATH}, f)
        print(f"  Modell + Daten fuer SHAP gespeichert unter: {task_dir}")

    return results

In [ ]:
# Zentral vorbereitete Daten laden (prepared_data.csv + split_meta.json)
print("="*70)
print("AUTO-SKLEARN BENCHMARK - GUTENBERG GAIT DATABASE (zentraler Split)")
print("="*70)
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nActive config:")
print(f"  CLASSIFICATION_TIME : {CLASSIFICATION_TIME}s")
print(f"  REGRESSION_TIME     : {REGRESSION_TIME}s")
print(f"  PER_RUN_TIME_LIMIT  : {PER_RUN_TIME_LIMIT}s")
print(f"  CV_FOLDS            : {CV_FOLDS}")
print(f"  MEMORY_LIMIT_MB     : {MEMORY_LIMIT_MB}MB")
print(f"  REGRESSION_DIRECTIONS: {REGRESSION_DIRECTIONS}")
print(f"  TARGET_PERCENTAGES  : {TARGET_PERCENTAGES}")
print(f"  EXCLUDE_WINDOW      : {EXCLUDE_WINDOW}")

df = pd.read_csv(PREPARED_DATA_PATH)
with open(SPLIT_META_PATH) as f:
    split_meta = json.load(f)

print(f"\nPrepared data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Split: {split_meta['n_subjects_train']} Train-Subjekte, "
      f"{split_meta['n_subjects_test']} Test-Subjekte "
      f"(erzeugt am {split_meta['created_at']}, random_state={split_meta['random_state']})")

required_cols = ['SUBJECT_ID', 'SPLIT', 'SEX_LABEL', 'AGE_BRACKET_LABEL', 'HEIGHT_BRACKET_LABEL']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Erwartete Spalten fehlen in {PREPARED_DATA_PATH}: {missing}. "
                      f"Bitte zuerst 00_preprocessing ausfuehren.")

In [ ]:
# Feature-Basis bauen: alle drei Kraftrichtungen kombiniert (V+AP+ML)
force_columns = []
for prefix in ['F_V_PRO_', 'F_AP_PRO_', 'F_ML_PRO_']:
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        raise ValueError(f"Keine Spalten mit Praefix '{prefix}' gefunden - bitte pruefen, "
                          f"ob 00_preprocessing mit dem AP/ML-Fix gelaufen ist.")
    force_columns.extend(cols)

print(f"Using {len(force_columns)} force features (V+AP+ML kombiniert)")

X = df[force_columns].values
valid_rows = ~np.isnan(X).any(axis=1)
X = X[valid_rows]
print(f"Feature matrix shape: {X.shape}")

df_valid = df[valid_rows].reset_index(drop=True)
groups_all = df_valid['SUBJECT_ID'].values
split_all = df_valid['SPLIT'].values
print(f"Anzahl eindeutiger Subjekte im gueltigen Datensatz: {df_valid['SUBJECT_ID'].nunique()}")

In [ ]:
# Klassifikations-Labels auslesen (bereits zentral kodiert in 00_preprocessing)
labels = {
    'sex': df_valid['SEX_LABEL'].values,
    'age_bracket_encoded': df_valid['AGE_BRACKET_LABEL'].values,
    'height_bracket_encoded': df_valid['HEIGHT_BRACKET_LABEL'].values,
}
print(f"Sex distribution:\n{df_valid['SEX'].value_counts().to_string()}")
print(f"Age bracket distribution:\n{df_valid['AGE_BRACKET'].value_counts().to_string()}")
print(f"Height bracket distribution:\n{df_valid['HEIGHT_BRACKET'].value_counts().to_string()}")

In [8]:
# Ergebnis-Grundstruktur anlegen
all_results = {
    'benchmark_info': {
        'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'n_samples': len(X),
        'n_features': X.shape[1],
        'n_subjects': int(df_valid['SUBJECT_ID'].nunique()),
        'framework': 'auto-sklearn',
        'config': {
            'classification_time': CLASSIFICATION_TIME,
            'regression_time': REGRESSION_TIME,
            'per_run_time_limit': PER_RUN_TIME_LIMIT,
            'cv_folds': CV_FOLDS,
            'memory_limit_mb': MEMORY_LIMIT_MB,
            'regression_directions': REGRESSION_DIRECTIONS,
            'target_percentages': TARGET_PERCENTAGES,
            'exclude_window': EXCLUDE_WINDOW,
            'split_strategy': 'precomputed, shared across frameworks (00_preprocessing.py)',
            'split_meta_source': SPLIT_META_PATH,
        }
    },
    'results': []
}

demo_feature_columns = force_columns

In [ ]:
# Klassifikation: Sex (aktiv)
print(f"\n{'#'*60}")
print(f"# CLASSIFICATION BENCHMARKS (auto-sklearn)")
print(f"{'#'*60}")

results = benchmark_autosklearn_classification(
    X, labels['sex'], groups_all, split_all, 'sex_classification',
    time_left_for_this_task=CLASSIFICATION_TIME,
    per_run_time_limit=PER_RUN_TIME_LIMIT,
    feature_names=demo_feature_columns,
)
all_results['results'].append(results)

In [ ]:
# Weitere Klassifikationsaufgaben
#
results = benchmark_autosklearn_classification(
    X, labels['age_bracket_encoded'], groups_all, split_all, 'age_bracket_classification',
    time_left_for_this_task=CLASSIFICATION_TIME,
    per_run_time_limit=PER_RUN_TIME_LIMIT,
    feature_names=demo_feature_columns,
)
all_results['results'].append(results)

results = benchmark_autosklearn_classification(
    X, labels['height_bracket_encoded'], groups_all, split_all, 'height_bracket_classification',
    time_left_for_this_task=CLASSIFICATION_TIME,
    per_run_time_limit=PER_RUN_TIME_LIMIT,
    feature_names=demo_feature_columns,
)
all_results['results'].append(results)

# Regression ueber alle Kraftkurven-Ziele (4 Punkte im Gangzyklus x 3 Richtungen)
force_results = []
for prefix in REGRESSION_DIRECTIONS:
    cols = sorted((c for c in df_valid.columns if c.startswith(prefix)),
                   key=lambda c: int(c[len(prefix):]))
    n = len(cols)
    for p in TARGET_PERCENTAGES:
        idx = min(max(int(round((p / 100.0) * (n - 1))), 0), n - 1)
        target_col = cols[idx]
        feature_cols = [c for c in cols if c not in
                         set(cols[max(0, idx - EXCLUDE_WINDOW): idx + EXCLUDE_WINDOW + 1])]
        if len(feature_cols) == 0:
            continue

        X_reg = df_valid[feature_cols].values
        y_reg = df_valid[target_col].values
        groups_reg = df_valid['SUBJECT_ID'].values
        split_reg = df_valid['SPLIT'].values

        valid_rows_reg = ~np.isnan(X_reg).any(axis=1) & ~np.isnan(y_reg)
        X_reg, y_reg = X_reg[valid_rows_reg], y_reg[valid_rows_reg]
        groups_reg, split_reg = groups_reg[valid_rows_reg], split_reg[valid_rows_reg]
        if len(X_reg) < 100:
            continue

        task_name = f"predict_{prefix.rstrip('_')}_{p}pct_{target_col}"
        try:
            results = benchmark_autosklearn_regression(
                X_reg, y_reg, groups_reg, split_reg, task_name,
                time_left_for_this_task=REGRESSION_TIME,
                per_run_time_limit=PER_RUN_TIME_LIMIT,
                feature_names=feature_cols,
            )
            results['n_feature_columns'] = len(feature_cols)
            results['exclude_window'] = EXCLUDE_WINDOW
            results['direction'] = prefix
            results['percent_of_cycle'] = p
            force_results.append(results)
        except ValueError as e:
            print(f"  Skipping {target_col}: no valid model found ({e})")
all_results['results'].extend(force_results)

In [ ]:
# Regression: demografische Variablen
print(f"\n{'#'*60}")
print(f"# REGRESSION BENCHMARKS - DEMOGRAPHIC VARIABLES")
print(f"{'#'*60}")

y_age = df_valid['AGE'].values
valid_age = ~np.isnan(y_age)
try:
    results = benchmark_autosklearn_regression(
        X[valid_age], y_age[valid_age], groups_all[valid_age], split_all[valid_age], 'age_regression',
        time_left_for_this_task=REGRESSION_TIME,
        per_run_time_limit=PER_RUN_TIME_LIMIT,
        feature_names=demo_feature_columns,
    )
    all_results['results'].append(results)
except ValueError as e:
    print(f"  Skipping age_regression: {e}")

y_height = df_valid['HEIGHT'].values
valid_height = ~np.isnan(y_height)
try:
    results = benchmark_autosklearn_regression(
        X[valid_height], y_height[valid_height], groups_all[valid_height], split_all[valid_height], 'height_regression',
        time_left_for_this_task=REGRESSION_TIME,
        per_run_time_limit=PER_RUN_TIME_LIMIT,
        feature_names=demo_feature_columns,
    )
    all_results['results'].append(results)
except ValueError as e:
    print(f"  Skipping height_regression: {e}")

In [ ]:
# Ergebnisse als JSON speichern
print(f"\n{'#'*60}")
print(f"# SAVING RESULTS")
print(f"{'#'*60}")

output_file = f'autosklearn_results_v4_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to: {output_file}")

In [ ]:
# Zusammenfassung ausgeben
print(f"\n{'='*70}")
print("BENCHMARK SUMMARY (auto-sklearn, precomputed subject-grouped split)")
print(f"{'='*70}")

tasks = {}
for res in all_results['results']:
    tasks.setdefault(res['task'], []).append(res)

for task, task_results in tasks.items():
    print(f"\n{task}:")
    for res in task_results:
        if res['task_type'] == 'classification':
            print(f"  Accuracy = {res['accuracy']:.4f}  "
                  f"Balanced_Acc = {res['balanced_accuracy']:.4f}  "
                  f"F1_weighted = {res['f1_weighted']:.4f}  "
                  f"Time = {res['training_time_seconds']:.1f}s")
        else:
            print(f"  R2 = {res['r2']:.4f}  "
                  f"MAE = {res['mae']:.4f}  RMSE = {res['rmse']:.4f}  "
                  f"Time = {res['training_time_seconds']:.1f}s")

print(f"\nTotal benchmark completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")